# Variant features

One row per lead variant of a qualifying disease credible set, aggregated over every credible
set that variant appears in: effect size, MAF, effective sample size, variance explained,
GERP constraint, VEP score, the concordance of effect direction across diseases, and the
diseases and therapeutic areas it is associated with. Methods "Variant-level pleiotropy
modelling".

These are the covariates the variant pleiotropy model is fitted on, and the source of the
directionality numbers and Supplementary Table 2.

Writes `variant_features`.

In [1]:
from gentropy.common.session import Session
from pyspark.sql import functions as f

from manuscript_methods import paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/19 00:28:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
credible_sets = session.spark.read.parquet(paper.derived("qualifying_credible_sets"))
# The variant-level analysis was built on the legacy hierarchy order, so the therapeutic-area
# count here uses that column (see 03_therapeutic_areas).
studies = session.spark.read.parquet(paper.derived("study_therapeutic_areas")).select(
    "studyId",
    f.col("mappedTherapeuticAreasLegacy").alias("mappedTherapeuticAreas"),
    "totalTherapeuticAreas",
    *paper.TA_COLUMNS.values(),
)
annotation = session.spark.read.parquet(paper.derived("study_annotation")).select(
    "studyId", "publicationDate", "effectiveSampleSize"
)
prioritised = (
    session.spark.read.parquet(paper.derived("prioritised_genes_per_cs"))
    .groupBy("studyLocusId")
    .agg(f.collect_list("geneId").alias("prioritisedGenes"))
)
variants = session.spark.read.parquet(paper.release("variant")).select(
    "variantId",
    f.filter("variantEffect", lambda x: x["method"] == "GERP")[0]["normalisedScore"].alias("gerpNormalised"),
    f.filter("variantEffect", lambda x: x["method"] == "VEP")[0]["score"].alias("vepScore"),
)
disease_names = session.spark.read.parquet(paper.release("disease") + "/disease.parquet").select("id", "name")

## Credible sets with their study covariates

In [3]:
rows = (
    credible_sets.join(prioritised, "studyLocusId", "left")
    .join(studies, "studyId", "inner")
    .join(annotation, "studyId", "inner")
    .withColumn("coefficientDetermination", f.col("variantStatistics.chi2Stat") / f.col("effectiveSampleSize"))
    .cache()
)
print("credible sets:", rows.count(), "| lead variants:", rows.select("variantId").distinct().count())

26/08/19 00:28:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


credible sets: 70618 | lead variants: 40706


## Aggregate to variants

`betaSignConcordance` is the share of associations agreeing on the more common direction of
effect, computed on the minor allele so that direction is comparable across studies.

In [4]:
minor_beta = f.col("rescaledStatistics.minorAlleleEstimatedBeta")
has_beta = f.col("originalBeta").isNotNull()
positive = f.sum(f.when(has_beta & (minor_beta > 0), 1.0).otherwise(0.0))
negative = f.sum(f.when(has_beta & (minor_beta < 0), 1.0).otherwise(0.0))
with_beta = f.sum(f.when(has_beta, 1.0).otherwise(0.0))

features = (
    rows.groupBy("variantId")
    .agg(
        f.mean(f.when(f.col("majorLdPopulation.ldPopulation") == "nfe", 1).otherwise(0)).alias("nonEURProportion"),
        f.min(f.abs(minor_beta)).alias("minAbsBeta"),
        f.max(f.abs(minor_beta)).alias("maxAbsBeta"),
        f.min("coefficientDetermination").alias("minCoefficientDetermination"),
        f.max("coefficientDetermination").alias("maxCoefficientDetermination"),
        f.min("effectiveSampleSize").alias("minEffectiveSampleSize"),
        f.max("effectiveSampleSize").alias("maxEffectiveSampleSize"),
        f.min("rescaledStatistics.varG").alias("minVarG"),
        f.max("rescaledStatistics.varG").alias("maxVarG"),
        f.max("majorLdPopulationMaf.value").alias("maxMAF"),
        f.avg(f.when(has_beta, f.signum(minor_beta))).alias("averageBetaSign"),
        f.stddev_pop(f.when(has_beta, f.signum(minor_beta))).alias("stdBetaSign"),
        f.greatest(positive / with_beta, negative / with_beta).alias("betaSignConcordance"),
        f.min("publicationDate").alias("earliestPublicationDate"),
        f.array_distinct(f.flatten(f.collect_list("diseaseIds"))).alias("diseaseIds"),
        f.size(f.array_distinct(f.flatten(f.collect_list("diseaseIds")))).alias("uniqueDiseases"),
        f.array_distinct(f.flatten(f.collect_list("mappedTherapeuticAreas"))).alias("therapeuticAreas"),
        f.size(f.array_distinct(f.flatten(f.collect_list("mappedTherapeuticAreas")))).alias("uniqueTherapeuticAreas"),
        f.array_distinct(f.flatten(f.collect_list("prioritisedGenes"))).alias("prioritisedGenes"),
        f.countDistinct("studyId").alias("totalStudies"),
        *[f.sum(column).alias(column) for column in paper.TA_COLUMNS.values()],
    )
    .join(variants, "variantId", "left")
)
features.write.mode("overwrite").parquet(paper.derived("variant_features"))

features = session.spark.read.parquet(paper.derived("variant_features"))
print("lead variants:", features.count())

lead variants: 40706


## Cross-check against the table the published Figure 3 was built from

In [5]:
columns = ["variantId", "maxAbsBeta", "maxMAF", "maxEffectiveSampleSize", "maxVarG", "gerpNormalised", "vepScore"]
new = features.select(columns).toPandas().drop_duplicates("variantId")
published = (
    session.spark.read.parquet(
        str(paper.ROOT / "chapters/_legacy/03-manuscript-figures/figure_3/python_scripts/variant_pleiotropy")
    )
    .select(columns)
    .toPandas()
    .drop_duplicates("variantId")
)
merged = new.merge(published, on="variantId", how="outer", suffixes=("_new", "_published"), indicator=True)
print(merged["_merge"].value_counts().to_string())
both = merged[merged["_merge"] == "both"]
for column in columns[1:]:
    delta = (both[f"{column}_new"] - both[f"{column}_published"]).abs().max()
    print(f"{column:<28} max abs difference: {delta:.3e}")

_merge
both          40706
left_only         0
right_only        0
maxAbsBeta                   max abs difference: 1.940e+00
maxMAF                       max abs difference: 0.000e+00
maxEffectiveSampleSize       max abs difference: 0.000e+00
maxVarG                      max abs difference: 0.000e+00
gerpNormalised               max abs difference: 0.000e+00
vepScore                     max abs difference: 0.000e+00
